# Prepare API Dataset

Ce notebook sert à :

1. reconstruire le dataset de features à partir des fichiers Home Credit,
2. isoler 100 clients de test pour l'API,
3. sauvegarder les fichiers CSV nécessaires,
4. vérifier la compatibilité du `model.joblib` avec les features exportées,
5. diagnostiquer un éventuel écart entre les features du modèle et celles de l'API.

> Remarque : les chemins sont centralisés dans la cellule **Paramètres** pour faciliter l'exécution en local.


## 1. Paramètres et imports

In [3]:

from pathlib import Path
import os
import gc
import numpy as np
import pandas as pd
import joblib
from pandas.api.types import is_string_dtype

# === A adapter si besoin ===
PROJECT_DIR = Path("/Users/macbook/OpenClassrooms/Projet7")
DATA_DIR = PROJECT_DIR / "Projet+Mise+en+prod+-+home-credit-default-risk"
API_DATA_DIR = PROJECT_DIR / "api" / "data"
MODEL_PATH = PROJECT_DIR / "model.joblib"

ID_COL = "SK_ID_CURR"
TARGET_COL = "TARGET"
N_SAMPLE = 100
RANDOM_STATE = 42

API_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("API_DATA_DIR:", API_DATA_DIR)
print("MODEL_PATH  :", MODEL_PATH)


PROJECT_DIR : /Users/macbook/OpenClassrooms/Projet7
DATA_DIR    : /Users/macbook/OpenClassrooms/Projet7/Projet+Mise+en+prod+-+home-credit-default-risk
API_DATA_DIR: /Users/macbook/OpenClassrooms/Projet7/api/data
MODEL_PATH  : /Users/macbook/OpenClassrooms/Projet7/model.joblib


## 2. Fonctions de feature engineering

In [5]:
def load_csv(name, nrows=None):
    return pd.read_csv(os.path.join(str(DATA_DIR), name), nrows=nrows)


def one_hot_encoder(df, nan_as_category=True):
    # Convert pandas string dtype -> object
    for c in df.columns:
        if is_string_dtype(df[c]):
            df[c] = df[c].astype("object")

    cat_cols = [c for c in df.columns if df[c].dtype == "object"]
    df = pd.get_dummies(df, columns=cat_cols, dummy_na=nan_as_category)
    return df


def application_base(nrows=None):
    train = load_csv("application_train.csv", nrows=nrows)
    test = load_csv("application_test.csv", nrows=nrows)

    df = pd.concat([train, test], axis=0, ignore_index=True)
    df = df[df["CODE_GENDER"] != "XNA"]

    # Remplacement anomalie
    df["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)

    # Ratios métier
    df["DAYS_EMPLOYED_PERC"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["INCOME_CREDIT_PERC"] = df["AMT_INCOME_TOTAL"] / df["AMT_CREDIT"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    df["ANNUITY_INCOME_PERC"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["PAYMENT_RATE"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]

    # Encodage
    df = one_hot_encoder(df, nan_as_category=True)
    gc.collect()
    return df


def bureau_and_balance(nrows=None):
    bureau = load_csv("bureau.csv", nrows=nrows)
    bb = load_csv("bureau_balance.csv", nrows=nrows)

    bb = one_hot_encoder(bb, nan_as_category=True)
    bureau = one_hot_encoder(bureau, nan_as_category=True)

    # Agrégation bb -> par SK_ID_BUREAU
    num_for_mean = [
        c for c in bb.columns
        if c not in ["SK_ID_BUREAU", "MONTHS_BALANCE"]
        and pd.api.types.is_numeric_dtype(bb[c])
    ]

    bb_aggregations = {"MONTHS_BALANCE": ["min", "max", "size"]}
    for c in num_for_mean:
        bb_aggregations[c] = ["mean"]

    bb_agg = bb.groupby("SK_ID_BUREAU").agg(bb_aggregations)
    bb_agg.columns = [f"{a}_{b.upper()}" for a, b in bb_agg.columns]

    # Agrégation bureau -> par SK_ID_CURR
    num_cols = [
        c for c in bureau.columns
        if c not in ["SK_ID_CURR"]
        and bureau[c].dtype != "uint8"
        and bureau[c].dtype != "int8"
    ]
    cat_cols = [
        c for c in bureau.columns
        if c not in ["SK_ID_CURR"]
        and (str(bureau[c].dtype).startswith("uint") or str(bureau[c].dtype).startswith("int8"))
    ]

    num_aggs = {c: ["min", "max", "mean", "sum"] for c in num_cols}
    cat_aggs = {c: ["mean"] for c in cat_cols}

    buro_agg = bureau.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    buro_agg.columns = [f"BURO_{a}_{b.upper()}" for a, b in buro_agg.columns]

    # Actifs
    if "CREDIT_ACTIVE_Active" in bureau.columns:
        active = bureau[bureau["CREDIT_ACTIVE_Active"] == 1]
        active_agg = active.groupby("SK_ID_CURR").agg(num_aggs)
        active_agg.columns = [f"ACTIVE_{a}_{b.upper()}" for a, b in active_agg.columns]
        buro_agg = buro_agg.join(active_agg, how="left")

    # Clos
    if "CREDIT_ACTIVE_Closed" in bureau.columns:
        closed = bureau[bureau["CREDIT_ACTIVE_Closed"] == 1]
        closed_agg = closed.groupby("SK_ID_CURR").agg(num_aggs)
        closed_agg.columns = [f"CLOSED_{a}_{b.upper()}" for a, b in closed_agg.columns]
        buro_agg = buro_agg.join(closed_agg, how="left")

    del bureau, bb, bb_agg
    gc.collect()
    return buro_agg


def previous_applications(nrows=None):
    prev = load_csv("previous_application.csv", nrows=nrows)
    prev = one_hot_encoder(prev, nan_as_category=True)

    for c in [
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION"
    ]:
        if c in prev.columns:
            prev[c].replace(365243, np.nan, inplace=True)

    prev["APP_CREDIT_PERC"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]

    num_cols = [
        "AMT_ANNUITY",
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "APP_CREDIT_PERC",
        "AMT_DOWN_PAYMENT",
        "AMT_GOODS_PRICE",
        "DAYS_DECISION",
        "CNT_PAYMENT"
    ]
    num_cols = [c for c in num_cols if c in prev.columns]

    num_aggs = {c: ["min", "max", "mean"] for c in num_cols}
    if "APP_CREDIT_PERC" in num_aggs:
        num_aggs["APP_CREDIT_PERC"] = ["min", "max", "mean", "var"]

    cat_cols = [
        c for c in prev.columns
        if c not in ["SK_ID_CURR", "SK_ID_PREV"] + num_cols
        and prev[c].dtype in [np.uint8, np.int8, np.int16, np.int32, np.int64]
    ]
    cat_aggs = {c: ["mean"] for c in cat_cols}

    prev_agg = prev.groupby("SK_ID_CURR").agg({**num_aggs, **cat_aggs})
    prev_agg.columns = [f"PREV_{a}_{b.upper()}" for a, b in prev_agg.columns]

    if "NAME_CONTRACT_STATUS_Approved" in prev.columns:
        approved = prev[prev["NAME_CONTRACT_STATUS_Approved"] == 1]
        approved_agg = approved.groupby("SK_ID_CURR").agg(num_aggs)
        approved_agg.columns = [f"APPROVED_{a}_{b.upper()}" for a, b in approved_agg.columns]
        prev_agg = prev_agg.join(approved_agg, how="left")

    if "NAME_CONTRACT_STATUS_Refused" in prev.columns:
        refused = prev[prev["NAME_CONTRACT_STATUS_Refused"] == 1]
        refused_agg = refused.groupby("SK_ID_CURR").agg(num_aggs)
        refused_agg.columns = [f"REFUSED_{a}_{b.upper()}" for a, b in refused_agg.columns]
        prev_agg = prev_agg.join(refused_agg, how="left")

    del prev
    gc.collect()
    return prev_agg


def pos_cash(nrows=None):
    pos = load_csv("POS_CASH_balance.csv", nrows=nrows)
    pos = one_hot_encoder(pos, nan_as_category=True)

    aggs = {
        "MONTHS_BALANCE": ["max", "mean", "size"],
        "SK_DPD": ["max", "mean"],
        "SK_DPD_DEF": ["max", "mean"]
    }

    pos_agg = pos.groupby("SK_ID_CURR").agg(aggs)
    pos_agg.columns = [f"POS_{a}_{b.upper()}" for a, b in pos_agg.columns]
    pos_agg["POS_COUNT"] = pos.groupby("SK_ID_CURR").size()

    del pos
    gc.collect()
    return pos_agg


def installments_payments(nrows=None):
    ins = load_csv("installments_payments.csv", nrows=nrows)
    ins = one_hot_encoder(ins, nan_as_category=True)

    ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

    aggs = {
        "DPD": ["max", "mean", "sum"],
        "DBD": ["max", "mean", "sum"],
        "PAYMENT_PERC": ["max", "mean", "var"],
        "PAYMENT_DIFF": ["max", "mean", "sum", "var"],
        "AMT_INSTALMENT": ["max", "mean", "sum"],
        "AMT_PAYMENT": ["min", "max", "mean", "sum"],
    }

    ins_agg = ins.groupby("SK_ID_CURR").agg(aggs)
    ins_agg.columns = [f"INSTAL_{a}_{b.upper()}" for a, b in ins_agg.columns]
    ins_agg["INSTAL_COUNT"] = ins.groupby("SK_ID_CURR").size()

    del ins
    gc.collect()
    return ins_agg


def credit_card_balance(nrows=None):
    cc = load_csv("credit_card_balance.csv", nrows=nrows)
    cc = one_hot_encoder(cc, nan_as_category=True)

    if "SK_ID_PREV" in cc.columns:
        cc.drop(columns=["SK_ID_PREV"], inplace=True)

    cc_agg = cc.groupby("SK_ID_CURR").agg(["min", "max", "mean", "sum"])
    cc_agg.columns = [f"CC_{a}_{b.upper()}" for a, b in cc_agg.columns]
    cc_agg["CC_COUNT"] = cc.groupby("SK_ID_CURR").size()

    del cc
    gc.collect()
    return cc_agg


def build_features(nrows=None):
    df = application_base(nrows=nrows)

    buro = bureau_and_balance(nrows=nrows)
    df = df.merge(buro, on="SK_ID_CURR", how="left")
    del buro
    gc.collect()

    prev = previous_applications(nrows=nrows)
    df = df.merge(prev, on="SK_ID_CURR", how="left")
    del prev
    gc.collect()

    pos = pos_cash(nrows=nrows)
    df = df.merge(pos, on="SK_ID_CURR", how="left")
    del pos
    gc.collect()

    ins = installments_payments(nrows=nrows)
    df = df.merge(ins, on="SK_ID_CURR", how="left")
    del ins
    gc.collect()

    cc = credit_card_balance(nrows=nrows)
    df = df.merge(cc, on="SK_ID_CURR", how="left")
    del cc
    gc.collect()

    return df

## 3. Reconstruction du dataset global

In [7]:

df = build_features(nrows=None)

print("Shape df :", df.shape)
display(df.head())


/var/folders/6q/vqmkpd597p36s9t_j9zm_k0c0000gn/T/ipykernel_22622/2234891041.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)
/var/folders/6q/vqmkpd597p36s9t_j9zm_k0c0000gn/T/ipykernel_22622/2234891041.py:110: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

Shape df : (356251, 954)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,CC_NAME_CONTRACT_STATUS_Signed_MIN,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_nan_MIN,CC_NAME_CONTRACT_STATUS_nan_MAX,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_NAME_CONTRACT_STATUS_nan_SUM,CC_COUNT
0,100002,1.0,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0.0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0.0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0.0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039.0,...,0.0,False,False,0.0,0.0,False,False,0.0,0.0,6.0
4,100007,0.0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Séparation train / test

In [9]:

train_df = df[df[TARGET_COL].notnull()].copy()
test_df = df[df[TARGET_COL].isnull()].copy()

print("train_df :", train_df.shape)
print("test_df  :", test_df.shape)
print(f"{ID_COL} présent dans test_df :", ID_COL in test_df.columns)
print(f"{TARGET_COL} présent dans test_df :", TARGET_COL in test_df.columns)


train_df : (307507, 954)
test_df  : (48744, 954)
SK_ID_CURR présent dans test_df : True
TARGET présent dans test_df : True


## 5. Création du dataset API (100 clients)

In [11]:

df_api = test_df.drop(columns=[TARGET_COL]).copy()
df_api_sample = df_api.sample(n=N_SAMPLE, random_state=RANDOM_STATE).copy()
X_api_sample = df_api_sample.drop(columns=[ID_COL]).copy()

print("df_api_sample :", df_api_sample.shape)
print("X_api_sample  :", X_api_sample.shape)
print(f"{ID_COL} présent dans X_api_sample ?", ID_COL in X_api_sample.columns)
display(df_api_sample.head())


df_api_sample : (100, 953)
X_api_sample  : (100, 952)
SK_ID_CURR présent dans X_api_sample ? False


,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,CC_NAME_CONTRACT_STATUS_Signed_MIN,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_nan_MIN,CC_NAME_CONTRACT_STATUS_nan_MAX,CC_NAME_CONTRACT_STATUS_nan_MEAN,CC_NAME_CONTRACT_STATUS_nan_SUM,CC_COUNT
322578,208550,2,450000.0,854896.5,36351.0,702000.0,0.032561,-11348,-1149.0,-5443.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
317664,173779,0,67500.0,296280.0,19062.0,225000.0,0.009175,-19852,NaN,-192.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
343820,365820,0,135000.0,339948.0,35694.0,315000.0,0.024610,-15138,-4891.0,-9273.0,...,0.0,False,False,0.0,0.0,False,False,0.0,0.0,14.0
313524,144092,0,256500.0,609898.5,31270.5,526500.0,0.002042,-15214,-2773.0,-2075.0,...,0.0,False,False,0.0,0.0,False,False,0.0,0.0,12.0
333826,291599,1,180000.0,450000.0,22977.0,450000.0,0.018029,-15806,-5816.0,-1188.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Sauvegarde des fichiers pour l'API

In [13]:

TEST_CLIENTS_PATH = API_DATA_DIR / "test_clients.csv"
TEST_FEATURES_PATH = API_DATA_DIR / "test_clients_features_only.csv"

df_api_sample.to_csv(TEST_CLIENTS_PATH, index=False)
X_api_sample.to_csv(TEST_FEATURES_PATH, index=False)

print("Fichiers sauvegardés :")
print("-", TEST_CLIENTS_PATH)
print("-", TEST_FEATURES_PATH)


Fichiers sauvegardés :
- /Users/macbook/OpenClassrooms/Projet7/api/data/test_clients.csv
- /Users/macbook/OpenClassrooms/Projet7/api/data/test_clients_features_only.csv


## 7. Vérification rapide des exports

In [15]:

df_check = pd.read_csv(TEST_CLIENTS_PATH)
X_check = pd.read_csv(TEST_FEATURES_PATH)

print("test_clients.csv               :", df_check.shape)
print("test_clients_features_only.csv :", X_check.shape)
print("Nb clients uniques            :", df_check[ID_COL].nunique())


test_clients.csv               : (100, 953)
test_clients_features_only.csv : (100, 952)
Nb clients uniques            : 100


## 8. Chargement du modèle

In [17]:

model = joblib.load(MODEL_PATH)

print("Type du modèle :", type(model))
print("Shape X_check  :", X_check.shape)


Type du modèle : <class 'xgboost.sklearn.XGBClassifier'>
Shape X_check  : (100, 952)


## 9. Diagnostic des types `object`

In [19]:

obj_cols = X_check.select_dtypes(include=["object"]).columns.tolist()

print("Nombre de colonnes object :", len(obj_cols))
print("Aperçu :", obj_cols[:20])


Nombre de colonnes object : 172
Aperçu : ['BURO_CREDIT_ACTIVE_Active_MIN', 'BURO_CREDIT_ACTIVE_Active_MAX', 'BURO_CREDIT_ACTIVE_Bad debt_MIN', 'BURO_CREDIT_ACTIVE_Bad debt_MAX', 'BURO_CREDIT_ACTIVE_Closed_MIN', 'BURO_CREDIT_ACTIVE_Closed_MAX', 'BURO_CREDIT_ACTIVE_Sold_MIN', 'BURO_CREDIT_ACTIVE_Sold_MAX', 'BURO_CREDIT_ACTIVE_nan_MIN', 'BURO_CREDIT_ACTIVE_nan_MAX', 'BURO_CREDIT_CURRENCY_currency 1_MIN', 'BURO_CREDIT_CURRENCY_currency 1_MAX', 'BURO_CREDIT_CURRENCY_currency 2_MIN', 'BURO_CREDIT_CURRENCY_currency 2_MAX', 'BURO_CREDIT_CURRENCY_currency 3_MIN', 'BURO_CREDIT_CURRENCY_currency 3_MAX', 'BURO_CREDIT_CURRENCY_currency 4_MIN', 'BURO_CREDIT_CURRENCY_currency 4_MAX', 'BURO_CREDIT_CURRENCY_nan_MIN', 'BURO_CREDIT_CURRENCY_nan_MAX']


In [20]:

for col in obj_cols[:10]:
    print(f"\n--- {col} ---")
    print(X_check[col].dropna().unique()[:10])



--- BURO_CREDIT_ACTIVE_Active_MIN ---
[False True]

--- BURO_CREDIT_ACTIVE_Active_MAX ---
[True False]

--- BURO_CREDIT_ACTIVE_Bad debt_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_Bad debt_MAX ---
[False]

--- BURO_CREDIT_ACTIVE_Closed_MIN ---
[False True]

--- BURO_CREDIT_ACTIVE_Closed_MAX ---
[True False]

--- BURO_CREDIT_ACTIVE_Sold_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_Sold_MAX ---
[False True]

--- BURO_CREDIT_ACTIVE_nan_MIN ---
[False]

--- BURO_CREDIT_ACTIVE_nan_MAX ---
[False]


## 10. Conversion des colonnes `object` en numérique

In [22]:

def coerce_object_columns_to_numeric(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    for col in object_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

X_check_fixed = coerce_object_columns_to_numeric(X_check)

print("Répartition des dtypes :")
print(X_check_fixed.dtypes.value_counts())

remaining_obj_cols = X_check_fixed.select_dtypes(include=["object"]).columns.tolist()
print("\nColonnes object restantes :", len(remaining_obj_cols))


Répartition des dtypes :
float64    759
bool       155
int64       38
Name: count, dtype: int64

Colonnes object restantes : 0


In [23]:

print("Nombre total de NaN :", X_check_fixed.isna().sum().sum())
print("Shape               :", X_check_fixed.shape)


Nombre total de NaN : 23962
Shape               : (100, 952)


## 11. Test du modèle sans ID

In [25]:

X_one = X_check_fixed.iloc[[0]].copy()

try:
    pred = model.predict(X_one)
    proba = model.predict_proba(X_one)
    print("Prediction :", pred)
    print("Probabilité :", proba)
except Exception as e:
    print("Erreur lors du test sans ID :")
    print(type(e).__name__, "-", e)


Erreur lors du test sans ID :
ValueError - Feature shape mismatch, expected: 953, got 952


## 12. Test du modèle avec l'ID inclus

Cette cellule permet de vérifier si le modèle a été entraîné **avec** `SK_ID_CURR`.


In [27]:

df_with_id = pd.read_csv(TEST_CLIENTS_PATH)
df_with_id_fixed = coerce_object_columns_to_numeric(df_with_id)

print("Avec ID :", df_with_id_fixed.shape)
print("Sans ID :", X_check_fixed.shape)


Avec ID : (100, 953)
Sans ID : (100, 952)


In [28]:

X_one_with_id = df_with_id_fixed.iloc[[0]].copy()

try:
    pred = model.predict(X_one_with_id)
    proba = model.predict_proba(X_one_with_id)
    print("Prediction :", pred)
    print("Probabilité :", proba)
except Exception as e:
    print("Erreur lors du test avec ID :")
    print(type(e).__name__, "-", e)


Prediction : [0]
Probabilité : [[0.983722   0.01627803]]
